- This file contains all the transformations that can be performed outside and before the fols of our cross validation algorithm.
- This correspond to the pre processing functions that will not induce data leackage.
- We decided to perform some of the safe operations outside the folds in order to make this project run more efficiently, by avoiding to reapply this step for all folds, we just do it once. 

In [1]:
import pandas as pd
import unicodedata
import re
from typing import Dict, Any
import numpy as np


Tranformations applied here:
- Year: minimum year and max year;
- Percentages within 0-100%;
- Dropping has damage and carID;
- 

## the upper, strip, erase weird characters, 

This function:
- keeps letters and digit;
- transforms full string to upper case;
- removes spaces in the middle, beggining and end; (might allow spaces in the middle)
- removes symbols and ponctuation; (might allow keeping -)
- removes acentos of the vogals.

In [ ]:
## replace categorical features with missing values with UNKNOWN

In [ ]:
def basic_string_transformer(word, 
    remove_middle_spaces: bool = True,
    allow_extra_chars: str = ""
    ):

    if pd.isna(word):
        return word
    
    s = str(word)

    # 1st: strip the leading and trailing whitespaces 
    s = s.strip()

    # 2nd: uppercase the string
    s = s.upper()

    # 3rd: removal of accents
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

    # 4th: remove middle spaces if required
    if remove_middle_spaces:
        s = s.replace(" ", "")
    else:
        s = re.sub(r"\s+", " ", s)
        s = s.strip()
    
    # 5th: remove ponctuation/symbols etc
    # basically, by default, only keeps A-Z characters and digits
    # optionally if the user selects extra allowed characters, we keep them too
    allowed = allow_extra_chars or "" 
    # if we want to keep middle spaces, we add space to allowed characters
    if not remove_middle_spaces:
        allowed += " "
    # pattern = f"[^A-Z0-9{re.escape(allowed)}]" we did not use this because treats it as a raw string - possible problem with new line 
    pattern = rf"[^A-Z0-9{re.escape(allowed)}]"
    s = re.sub(pattern, "", s)

    #6th: if string becomes empty or only has spaces, return NaN
    if s.strip() == "":
        return np.nan
    
    return s

In [ ]:
def def_string_basic_transformer(
    df: pd.DataFrame,
    column: str,
    remove_middle_spaces: bool = True,
    allow_extra_chars: str = ""
    ) -> pd.DataFrame: #returns a dataframe
    """ 
    Apply basic string normalization to a given column of a DataFrame.

    - Parameters: 
    df : pd.DataFrame
        Input DataFrame.
    column : str
        Name of the column to transform.
    remove_inner_spaces : bool, default True
        If True, remove all spaces (including in the middle).
        If False, keep inner spaces (normalizing multiple spaces to just one).
    allow_extra_chars : str, default ""
        String of extra characters to keep (e.g., "-/" to keep '-' and '/').

    - Returns
    pd.DataFrame
        A copy of df with the specified column transformed.
    """
    
    df_out = df.copy()
    df_out[column] = df_out[column].apply(
        lambda x: basic_string_transformer(
            x,      
            remove_middle_spaces=remove_middle_spaces,
            allow_extra_chars=allow_extra_chars,
        )
    )   
    df_out[column] = df_out[column].astype("string")

    return df_out

In [ ]:
valid_brands = ['FORD', 'MERCEDES', 'VW', 'OPEL', 'BMW', 'AUDI', 'TOYOTA', 'SKODA', 'HYUNDAI']

# Fill missing Brand values with 'UNKNOWN'
train['Brand'] = train['Brand'].fillna('UNKNOWN') 

# Identify brands in the dataset that are not in the list of valid brands
invalids = sorted(
    [b for b in train['Brand'].unique() if b not in valid_brands],
    key=len
)

In [ ]:
def correct_invalid_brands_in_df(df, col, valid_brands, invalids):
    # Dictionary to store corrections applied
    corrections = {}
    
    for invalid in invalids:
        if invalid == 'UNKNOWN':
            continue  # Skip UNKNOWN values
        
        # Check if the invalid brand is contained in exactly one valid brand
        matches = [vb for vb in valid_brands if invalid in vb]
        
        if len(matches) == 1:
            valid = matches[0]
            df.loc[df[col] == invalid, col] = valid  # Replace invalid with the valid brand
            corrections[invalid] = valid

    # Identify any remaining invalid values that were not corrected
    remaining_invalids = [
        b for b in df[col].unique() 
        if b not in valid_brands and b not in corrections.keys()
    ]
    
    return df, corrections, remaining_invalids


# Cross val functions

For the transformations to be performed inside the cross validation folds, we followed a fit and transform logic.
- Fit: the fit functions are the ones that are consist of imputations from the dataset, basically calculations such as median, average, etc. This are calculated based on the training dataset.
- Transform: this consist on the transformations that eiter don't rely on calculations or that apply the previously "fit" imputations made (for example, the actual replacement of missing values with the median). 

in total we have 4 catgeorical features, each of them with their own transformations:
1) brand:
    - correct_invalid_brands_in_df -> transformm 
    - correct_ambiguous_brands -> problem: envolves both fitting and transforming. 2 options:
        - aplly outside fold and say its just a semantic correction;
        - split in 2 fit and transform and store the state
    
2) model
    - correct_invalid_models: same problem with split of fitting and transforming

3) fuelType
    - correct_invalid_fueltypes

4) transmission
    - correct_invalid_transmissions 

## 1) Brand

In [ ]:
valid_brands 

In [2]:
def _choose_brand_from_counts(counts: pd.Series, invalid_brand: str | None) -> str | None:
    """
    Given brand frequency counts for a group (e.g., a specific model or year)
    and the original invalid brand string, select a single brand according to:

    1. Take the highest frequency (the mode).
    2. If several brands tie for the highest frequency:
       - If exactly one of those brands contains the invalid brand as a
         substring, choose that one.
       - Otherwise, choose the first brand among the tied ones (i.e., the
         first in the value_counts order).

    If counts is empty, returns None.
    """
    if counts is None or len(counts) == 0:
        return None

    # value_counts is already sorted by frequency (desc) by default
    max_count = counts.iloc[0]
    top_brands = counts[counts == max_count].index.tolist()

    if len(top_brands) == 1:
        # Unique mode
        return top_brands[0]

    # There is a tie: try to break it using substring of the invalid brand
    if invalid_brand is None:
        invalid_upper = ""
    else:
        invalid_upper = str(invalid_brand).upper()

    substring_matches = [b for b in top_brands if invalid_upper and invalid_upper in str(b).upper()]

    if len(substring_matches) == 1:
        # Exactly one candidate contains the invalid brand string
        return substring_matches[0]

    # Either no substring match, or more than one → fall back to the first top brand
    return top_brands[0]


In [3]:
def fit_ambiguous_brand_resolver(
    train_df: pd.DataFrame,
    valid_brands: list[str],
    brand_col: str = "Brand",
    model_col: str = "model",
    year_col: str = "year",
) -> Dict[str, Any]:
    """
    Fit step for resolving ambiguous/invalid brands.

    This function learns, from the training data only:

    - The global most frequent valid brand (overall mode).
    - For each model, the frequency distribution of valid brands.
    - For each year, the frequency distribution of valid brands.

    These statistics are later used by the transform step to decide, for each
    invalid/ambiguous brand, which valid brand to choose.

    Assumptions:
    - `brand_col` has already been normalized (uppercased, accents removed, etc.).
    - `valid_brands` contains the canonical brand names that should be kept.
    """
    tmp = train_df.copy()

    valid_set = set(valid_brands)

    # Filter to rows where brand is valid
    valid_mask = tmp[brand_col].isin(valid_set)

    # Global most frequent valid brand
    brand_freq = tmp.loc[valid_mask, brand_col].value_counts()
    global_most_common_brand = brand_freq.index[0] if not brand_freq.empty else None

    # Brand frequency per year (only using valid brands)
    brand_counts_by_year: Dict[Any, pd.Series] = {}
    if year_col in tmp.columns:
        grouped_year = tmp.loc[valid_mask].groupby(year_col)[brand_col]
        for year_val, series in grouped_year:
            # series is the brand column for that year
            counts = series.value_counts()
            brand_counts_by_year[year_val] = counts

    # Brand frequency per model (only using valid brands)
    brand_counts_by_model: Dict[Any, pd.Series] = {}
    if model_col in tmp.columns:
        grouped_model = tmp.loc[valid_mask].groupby(model_col)[brand_col]
        for model_val, series in grouped_model:
            counts = series.value_counts()
            brand_counts_by_model[model_val] = counts

    state = {
        "brand_col": brand_col,
        "model_col": model_col,
        "year_col": year_col,
        "valid_brands": valid_set,
        "global_most_common_brand": global_most_common_brand,
        "brand_counts_by_year": brand_counts_by_year,
        "brand_counts_by_model": brand_counts_by_model,
    }

    return state


In [4]:
def transform_ambiguous_brands(
    df: pd.DataFrame,
    state: Dict[str, Any],
) -> tuple[pd.DataFrame, dict, list]:
    """
    Transform step for resolving ambiguous/invalid brands using
    statistics learned in `fit_ambiguous_brand_resolver`.

    For each row where the brand is not in `valid_brands`:

    - If `model` is known (and not 'UNKNOWN'):
        1. Look at the frequency distribution of valid brands for that model
           (from the training data).
        2. Choose the brand according to `_choose_brand_from_counts`.
        3. If no information is available for that model:
           - If original brand is 'UNKNOWN' -> use global most frequent valid brand.
           - Otherwise -> use the last replacement used for that invalid brand
             (past_replacements) or the global most frequent brand if none.

    - If `model` is missing or 'UNKNOWN':
        1. If both `year` is missing/empty AND original brand is 'UNKNOWN':
           -> directly use the global most frequent valid brand.
        2. Else, if `year` is present and we have statistics for that year:
           -> choose brand using `_choose_brand_from_counts`.
        3. If that still fails, use the global most frequent valid brand.

    Returns
    -------
    df_out : pd.DataFrame
        A copy of `df` with the brand column corrected.
    corrections : dict
        Mapping from (original_brand, model, year) to the chosen corrected brand.
    still_invalid : list
        List of brands that remain not in `valid_brands` and not equal to 'UNKNOWN'.
    """
    brand_col = state["brand_col"]
    model_col = state["model_col"]
    year_col = state["year_col"]
    valid_brands = state["valid_brands"]
    global_most_common_brand = state["global_most_common_brand"]
    brand_counts_by_year = state["brand_counts_by_year"]
    brand_counts_by_model = state["brand_counts_by_model"]

    df_out = df.copy()

    # Ensure no missing brands become a problem: treat NaN as 'UNKNOWN'
    df_out[brand_col] = df_out[brand_col].fillna("UNKNOWN")

    corrections: dict = {}
    past_replacements: dict = {}

    for idx, row in df_out.iterrows():
        original_brand = row[brand_col]
        brand_upper = str(original_brand).upper() if pd.notna(original_brand) else "UNKNOWN"

        # Skip already valid brands
        if brand_upper in valid_brands:
            continue

        model = row.get(model_col, None)
        year = row.get(year_col, None)

        corrected = None

        # ---------- Case 1: model is unknown or missing ----------
        if pd.isna(model) or str(model).strip().upper() == "UNKNOWN":

            # Special case: brand == 'UNKNOWN' and year is missing/empty
            if (pd.isna(year) or str(year).strip() == "") and brand_upper == "UNKNOWN":
                corrected = global_most_common_brand

            else:
                # Try using year-based statistics first
                if pd.notna(year) and year in brand_counts_by_year:
                    counts = brand_counts_by_year[year]
                    corrected = _choose_brand_from_counts(counts, brand_upper)

                # If still nothing, fall back to global most common brand
                if corrected is None:
                    corrected = global_most_common_brand

        # ---------- Case 2: model is known ----------
        else:
            if model in brand_counts_by_model:
                counts = brand_counts_by_model[model]
                corrected = _choose_brand_from_counts(counts, brand_upper)

            # No statistics available for this model
            if corrected is None:
                if brand_upper == "UNKNOWN":
                    # For UNKNOWN, go straight to global fallback
                    corrected = global_most_common_brand
                else:
                    # Try to be consistent with previous replacements for this invalid
                    corrected = past_replacements.get(brand_upper, global_most_common_brand)

        # Apply the correction
        df_out.at[idx, brand_col] = corrected
        corrections[(original_brand, model, year)] = corrected
        past_replacements[brand_upper] = corrected

        # Optional: logging (comment out if not needed)
        # if corrected == "UNKNOWN":
        #     print(f"FINAL RESULT: '{original_brand}' (model='{model}', year='{year}') - remained 'UNKNOWN'")
        # else:
        #     print(f"FINAL RESULT: '{original_brand}' (model='{model}', year='{year}') - corrected to '{corrected}'")

    # Brands that are still not valid (and not 'UNKNOWN') after correction
    still_invalid = [
        b for b in df_out[brand_col].unique()
        if b not in valid_brands and b != "UNKNOWN"
    ]

    return df_out, corrections, still_invalid


### Model


# 2 Numerical features

In [5]:
# basic function before cross val 

def basic_num_prep(
        series: pd.Series,
        clip_max: float | None = None,
        min_value: float | None = None,
        do_abs: bool = True,
        flooring: bool = False,
        rounding: bool = False,
        round_decimals: int = 0,
) -> pd.Series:
    
    """
    """ 
    
    # ensures we are working with numeric data
    s = pd.to_numeric(series, errors="coerce").copy()

    # absolute value transformation 
    if do_abs:
        s = s.abs()
    
    # cliiping maximum
    if clip_max is not None:
        s = s.clip(upper=clip_max)
    
    # enforce minimum
    if min_value is not None:
        s = s.clip(lower=min_val)
    
    if flooring:
        s = np.floor(s)
    elif rounding:
        s = s.round(round_decimals)
    
    return s

In [ ]:
# dropping carid and has damage
#....

# inside each fold


### year

In [ ]:
# Year 
# fit que calcula a mediana agrupada por model e caso seja nula
# usa a mediana global

#transform: aplica a median; as outras transformações serão feitas antes no coiso

def fit_year_median(
        train_df: pd.DataFrame,
        year_col: str = "year",
        model_col: str = "model",
        max_valid_year: int = 2020,
):
    tmp = train_df.copy()

    # ensure numeric 
    tmp[year_col] = pd.to_numeric(tmp[year_col], errors="coerce")

    # cap to maximum allowed year, already performed before but for safety 
    tmp.loc[tmp[year_col] > max_valid_year, year_col] = max_valid_year

    # floor to int already done before too
    tmp[year_col] = np.floor(tmp[year_col])

    # computed medians
    global_year_median = tmp[year_col].median()
    model_year_median = tmp.groupby(model_col)[year_col].median()

    state = {
        "year_col": year_col,
        "model_col": model_col,
        "max_valid_year":  max_valid_year,
        "global_year_median": global_year_median,
        "model_year_median": model_year_median
    }

    return state


In [7]:
def transform_year_with_model_median(
        df: pd.DataFrame,
        state: dict,
) -> pd.DataFrame:
    """
    """
    year_col = state["year_col"]
    model_col = state["model_col"]
    max_valid_year = state["max_valid_year"]
    global_median = state["global_year_median"]
    model_median = state["model_year_median"]

    df_out = df.copy()

    # 1) Ensure numeric
    df_out[year_col] = pd.to_numeric(df_out[year_col], errors="coerce")

    # 2) Cap to max_valid_year
    df_out.loc[df_out[year_col] > max_valid_year, year_col] = max_valid_year

    # 3) Floor to integer
    df_out[year_col] = np.floor(df_out[year_col])

    # 4) Impute missing years:
    #    - first use model-specific median
    df_out[year_col] = df_out[year_col].fillna(
        df_out[model_col].map(model_median)
    )
    #    - then use global median as fallback
    df_out[year_col] = df_out[year_col].fillna(global_median)

    return df_out

### previousOwners

In [ ]:
def fit_previous_owners_imputer(
    train_df: pd.DataFrame,
    owners_col: str = "previousOwners",
    year_col: str = "year",
    mileage_col: str = "mileage",
    mileage_threshold: float = 15000.0,
    min_years_since_registration: float = 2.0,
):
    """
    Fit step for previousOwners preprocessing.

    Assumes that `year` and `mileage` have already been preprocessed.

    Steps on train_df:
    1. Convert previousOwners to numeric (invalid -> NaN).
    2. Take absolute value (negatives become positive).
    3. Round to nearest integer.
    4. Apply "zero imprecision correction":
       If ((max_year - year) > min_years_since_registration OR mileage > mileage_threshold)
       AND previousOwners == 0 -> set previousOwners = 1.
    5. Compute the median of previousOwners (after the above cleaning, before imputation).

    Returns
    -------
    state : dict
        Dictionary containing all parameters needed for the transform step:
        - 'owners_col': name of previous owners column
        - 'year_col': name of year column
        - 'mileage_col': name of mileage column
        - 'max_year': max year observed in train
        - 'mileage_threshold': threshold used for mileage condition
        - 'min_years_since_registration': threshold for age condition
        - 'imputation_median': median of cleaned previousOwners (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[owners_col] = pd.to_numeric(tmp[owners_col], errors="coerce")

    # 2) Absolute values
    tmp[owners_col] = tmp[owners_col].abs()

    # 3) Round to nearest integer
    tmp[owners_col] = tmp[owners_col].round()

    # Compute max_year based on train only
    max_year = tmp[year_col].max()

    # 4) Zero imprecision correction
    #    If car is "old enough" or has enough mileage, a 0-owners entry is suspicious.
    #    Condition: ((max_year - year) > min_years_since_registration) OR (mileage > mileage_threshold)
    #               AND previousOwners == 0
    inac_own_condition_train = (
        ((max_year - tmp[year_col]) > min_years_since_registration)
        | (tmp[mileage_col] > mileage_threshold)
    ) & (tmp[owners_col] == 0)

    tmp.loc[inac_own_condition_train, owners_col] = 1

    # 5) Compute median after cleaning (NaNs are skipped)
    imputation_median = tmp[owners_col].median(skipna=True)


    state = {
        "owners_col": owners_col,
        "year_col": year_col,
        "mileage_col": mileage_col,
        "max_year": max_year,
        "mileage_threshold": mileage_threshold,
        "min_years_since_registration": min_years_since_registration,
        "imputation_median": imputation_median,
    }

    return state



def transform_previous_owners_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for previousOwners preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert previousOwners to numeric (invalid -> NaN).
    2. Take absolute value (negatives become positive).
    3. Round to nearest integer.
    4. Apply "zero imprecision correction" using the `max_year` learned in fit:
       If ((max_year - year) > min_years_since_registration OR mileage > mileage_threshold)
       AND previousOwners == 0 -> set previousOwners = 1.
    5. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_previous_owners_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the previousOwners column cleaned and imputed.
    """
    owners_col = state["owners_col"]
    year_col = state["year_col"]
    mileage_col = state["mileage_col"]
    max_year = state["max_year"]
    mileage_threshold = state["mileage_threshold"]
    min_years_since_registration = state["min_years_since_registration"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[owners_col] = pd.to_numeric(df_out[owners_col], errors="coerce")

    # 2) Absolute values
    df_out[owners_col] = df_out[owners_col].abs()

    # 3) Round to nearest integer
    df_out[owners_col] = df_out[owners_col].round()

    # 4) Zero imprecision correction (using max_year from TRAIN ONLY)
    inac_own_condition = (
        ((max_year - df_out[year_col]) > min_years_since_registration)
        | (df_out[mileage_col] > mileage_threshold)
    ) & (df_out[owners_col] == 0)

    df_out.loc[inac_own_condition, owners_col] = 1

    # 5) Impute NaNs with train-based median
    df_out[owners_col] = df_out[owners_col].fillna(imputation_median)

    return df_out

### Paint quality

In [9]:
def fit_paint_quality_imputer(
    train_df: pd.DataFrame,
    paint_col: str = "paintQuality%",
    clip_max: float = 100.0,
    do_abs: bool = True,
):
    """
    Fit step for paintQuality% preprocessing.

    Steps applied on train_df:
    1. Convert paint quality column to numeric (invalid values -> NaN).
    2. Optionally take absolute value (negatives become positive).
    3. Clip values above `clip_max` to `clip_max`.
    4. Compute the median of the cleaned paint quality column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    paint_col : str, default "paintQuality%%"
        Name of the paint quality column.
    clip_max : float, default 100.0
        Maximum allowed value for paint quality.
    do_abs : bool, default True
        If True, apply absolute value to the column.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'paint_col': name of paint quality column
        - 'clip_max': clipping value used
        - 'do_abs': whether abs() was applied
        - 'imputation_median': median of cleaned paint quality (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[paint_col] = pd.to_numeric(tmp[paint_col], errors="coerce")

    # 2) Absolute values (optionally)
    if do_abs:
        tmp[paint_col] = tmp[paint_col].abs()

    # 3) Clip upper bound
    tmp[paint_col] = tmp[paint_col].clip(upper=clip_max)

    # 4) Median after cleaning (NaNs skipped)
    imputation_median = tmp[paint_col].median(skipna=True)

    state = {
        "paint_col": paint_col,
        "clip_max": clip_max,
        "do_abs": do_abs,
        "imputation_median": imputation_median,
    }

    return state

In [10]:
def transform_paint_quality_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for paintQuality% preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert paint quality column to numeric (invalid values -> NaN).
    2. Optionally take absolute value (depending on `state['do_abs']`).
    3. Clip values above `clip_max` (from fit).
    4. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_paint_quality_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the paint quality column cleaned and imputed.
    """
    paint_col = state["paint_col"]
    clip_max = state["clip_max"]
    do_abs = state["do_abs"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[paint_col] = pd.to_numeric(df_out[paint_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[paint_col] = df_out[paint_col].abs()

    # 3) Clip upper bound
    df_out[paint_col] = df_out[paint_col].clip(upper=clip_max)

    # 4) Impute NaNs with train-based median
    df_out[paint_col] = df_out[paint_col].fillna(imputation_median)

    return df_out


### mileage

In [11]:
def fit_mileage_imputer(
    train_df: pd.DataFrame,
    mileage_col: str = "mileage",
    do_abs: bool = True,
):
    """
    Fit step for mileage preprocessing.

    Steps applied on train_df:
    1. Convert mileage column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Compute the median of the cleaned mileage column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    mileage_col : str, default "mileage"
        Name of the mileage column.
    do_abs : bool, default True
        If True, apply absolute value to the mileage column.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'mileage_col': name of mileage column
        - 'do_abs': whether abs() was applied
        - 'imputation_median': median of cleaned mileage (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric (invalid values -> NaN)
    tmp[mileage_col] = pd.to_numeric(tmp[mileage_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[mileage_col] = tmp[mileage_col].abs()

    # 3) Median after cleaning (NaNs skipped)
    imputation_median = tmp[mileage_col].median(skipna=True)

    state = {
        "mileage_col": mileage_col,
        "do_abs": do_abs,
        "imputation_median": imputation_median,
    }

    return state


In [ ]:
def transform_mileage_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for mileage preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert mileage column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_mileage_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the mileage column cleaned and imputed.
    """
    mileage_col = state["mileage_col"]
    do_abs = state["do_abs"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[mileage_col] = pd.to_numeric(df_out[mileage_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[mileage_col] = df_out[mileage_col].abs()

    # 3) Impute NaNs with train-based median
    df_out[mileage_col] = df_out[mileage_col].fillna(imputation_median)

    return df_out

### tax THIS ONE NEEDS FURTHER IMPROVEMENT

In [13]:
def fit_tax_imputer(
    train_df: pd.DataFrame,
    tax_col: str = "tax",
    do_abs: bool = True,
):
    """
    Fit step for tax preprocessing.

    Steps applied on train_df:
    1. Convert tax column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Compute the median of the cleaned tax column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    tax_col : str, default "tax"
        Name of the tax column.
    do_abs : bool, default True
        If True, apply absolute value to the tax column.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'tax_col': name of tax column
        - 'do_abs': whether abs() was applied
        - 'imputation_median': median of cleaned tax (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric (invalid values -> NaN)
    tmp[tax_col] = pd.to_numeric(tmp[tax_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[tax_col] = tmp[tax_col].abs()

    # 3) Median after cleaning (NaNs skipped)
    imputation_median = tmp[tax_col].median(skipna=True)

    state = {
        "tax_col": tax_col,
        "do_abs": do_abs,
        "imputation_median": imputation_median,
    }

    return state


In [ ]:
def transform_tax_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for tax preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert tax column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_tax_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the tax column cleaned and imputed.
    """
    tax_col = state["tax_col"]
    do_abs = state["do_abs"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[tax_col] = pd.to_numeric(df_out[tax_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[tax_col] = df_out[tax_col].abs()

    # 3) Impute NaNs with train-based median
    df_out[tax_col] = df_out[tax_col].fillna(imputation_median)

    return df_out

### mpg

I think this one could be improved too

In [15]:
def fit_mpg_imputer(
    train_df: pd.DataFrame,
    mpg_col: str = "mpg",
    do_abs: bool = True,
    clip_lower: float = 10.0,
    clip_upper: float = 200.0,
):
    """
    Fit step for mpg preprocessing.

    Steps applied on train_df:
    1. Convert mpg column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Compute the median of the cleaned mpg column (train only),
       BEFORE any clipping.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    mpg_col : str, default "mpg"
        Name of the mpg column.
    do_abs : bool, default True
        If True, apply absolute value to the mpg column.
    clip_lower : float, default 10.0
        Lower bound used later for clipping (stored for consistency).
    clip_upper : float, default 200.0
        Upper bound used later for clipping (stored for consistency).

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'mpg_col': name of mpg column
        - 'do_abs': whether abs() was applied
        - 'clip_lower': lower bound for clipping
        - 'clip_upper': upper bound for clipping
        - 'imputation_median': median of cleaned mpg (train only, pre-clip)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[mpg_col] = pd.to_numeric(tmp[mpg_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[mpg_col] = tmp[mpg_col].abs()

    # 3) Median before clipping
    imputation_median = tmp[mpg_col].median(skipna=True)

    state = {
        "mpg_col": mpg_col,
        "do_abs": do_abs,
        "clip_lower": clip_lower,
        "clip_upper": clip_upper,
        "imputation_median": imputation_median,
    }

    return state

In [16]:
def transform_mpg_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for mpg preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):

    1. Convert mpg column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Impute NaNs with the median learned from train.
    4. Clip mpg values between clip_lower and clip_upper (inclusive).

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_mpg_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the mpg column cleaned, imputed and clipped.
    """
    mpg_col = state["mpg_col"]
    do_abs = state["do_abs"]
    clip_lower = state["clip_lower"]
    clip_upper = state["clip_upper"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[mpg_col] = pd.to_numeric(df_out[mpg_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[mpg_col] = df_out[mpg_col].abs()

    # 3) Impute NaNs with train-based median
    df_out[mpg_col] = df_out[mpg_col].fillna(imputation_median)

    # 4) Clip to the specified range
    df_out[mpg_col] = df_out[mpg_col].clip(lower=clip_lower, upper=clip_upper)

    return df_out


### engine size

In [ ]:
def fit_engine_size_imputer(
    train_df: pd.DataFrame,
    engine_col: str = "engineSize",
    do_abs: bool = True,
    treat_zero_as_nan: bool = True,
):
    """
    Fit step for engineSize preprocessing.

    Steps applied on train_df:
    1. Convert engine size column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Optionally treat 0 values as NaN (so they are imputed).
    4. Compute the median of the cleaned engine size column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    engine_col : str, default "engineSize"
        Name of the engine size column.
    do_abs : bool, default True
        If True, apply absolute value to the engine size column.
    treat_zero_as_nan : bool, default True
        If True, replace exact zeros with NaN before computing the median.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'engine_col': name of engine size column
        - 'do_abs': whether abs() was applied
        - 'treat_zero_as_nan': whether 0 was treated as NaN
        - 'imputation_median': median of cleaned engine size (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[engine_col] = pd.to_numeric(tmp[engine_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[engine_col] = tmp[engine_col].abs()

    # 3) Treat zeros as NaN (if requested)
    if treat_zero_as_nan:
        tmp.loc[tmp[engine_col] == 0, engine_col] = np.nan

    # 4) Median after cleaning (NaNs skipped)
    imputation_median = tmp[engine_col].median(skipna=True)

    state = {
        "engine_col": engine_col,
        "do_abs": do_abs,
        "treat_zero_as_nan": treat_zero_as_nan,
        "imputation_median": imputation_median,
    }

    return state

In [18]:
def transform_engine_size_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for engineSize preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):

    1. Convert engine size column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Optionally treat 0 values as NaN (same rule as in fit).
    4. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_engine_size_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the engine size column cleaned and imputed.
    """
    engine_col = state["engine_col"]
    do_abs = state["do_abs"]
    treat_zero_as_nan = state["treat_zero_as_nan"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[engine_col] = pd.to_numeric(df_out[engine_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[engine_col] = df_out[engine_col].abs()

    # 3) Treat zeros as NaN (if used in fit)
    if treat_zero_as_nan:
        df_out.loc[df_out[engine_col] == 0, engine_col] = np.nan

    # 4) Impute NaNs with train-based median
    df_out[engine_col] = df_out[engine_col].fillna(imputation_median)

    return df_out
